# Chấm điểm động tác chống đẩy — Bước 1: trích keypoint

Pipeline tổng thể:

`144 video → MediaPipe Pose → keypoint (T, 33, 4) → đếm rep + cắt rep → chuẩn hoá → LSTM / ST-GCN → đúng / sai`

Notebook này làm phần đầu: tải dữ liệu, trích keypoint cho toàn bộ video chống đẩy và lưu ra `.npz`.

## 1. Cấu hình đường dẫn

In [ ]:
import os

WORK_DIR = "/kaggle/working"
ZIP_PATH = f"{WORK_DIR}/dataset.zip"
DATASET_DIR = f"{WORK_DIR}/dataset"
PUSHUP_DIR = f"{DATASET_DIR}/Dataset Exercise Quality-wise/push_up"
MODEL_PATH = f"{WORK_DIR}/pose_landmarker_full.task"
OUT_DIR = f"{WORK_DIR}/pushup_keypoints"

DATASET_URL = "https://data.mendeley.com/public-files/datasets/kgbb3yn47p/files/2499db21-0988-42a4-b068-0e79a8a3c51a/file_downloaded"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"

## 2. Tải dataset Mendeley và model pose

Cell này chạy lại được: cái gì đã có thì bỏ qua.

In [ ]:
if not os.path.isdir(PUSHUP_DIR):
    if not os.path.exists(ZIP_PATH):
        !wget -q --show-progress -O "{ZIP_PATH}" "{DATASET_URL}"
    !unzip -q -o "{ZIP_PATH}" -d "{DATASET_DIR}"
    os.remove(ZIP_PATH)   # giải phóng 2,9 GB

if not os.path.exists(MODEL_PATH):
    !wget -q -O "{MODEL_PATH}" "{MODEL_URL}"

print("Dataset:", os.path.isdir(PUSHUP_DIR), "| Model:", os.path.exists(MODEL_PATH))

## 3. Cài MediaPipe

In [ ]:
!pip install -q mediapipe

import mediapipe as mp
print("MediaPipe:", mp.__version__)

## 4. Thống kê video chống đẩy

In [ ]:
for quality in ["good", "bad"]:
    for view in ["front", "side", "diagonal"]:
        n = len(os.listdir(os.path.join(PUSHUP_DIR, quality, view)))
        print(f"{quality}/{view}: {n} video")

## 5. Trích keypoint cho toàn bộ video

Mỗi video lưu thành một file `.npz` gồm:
- `kp2d` `(T, 33, 4)`: x, y, z, visibility theo toạ độ ảnh đã chuẩn hoá về [0, 1]
- `kp3d` `(T, 33, 4)`: `pose_world_landmarks` (mét, gốc tại hông)
- `detected` `(T,)`: khung hình có phát hiện được người hay không
- `fps`, `width`, `height`

Chạy lại cell này sẽ bỏ qua các video đã trích xong.

In [ ]:
import re, glob, json, time
import cv2
import numpy as np
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

os.makedirs(os.path.join(OUT_DIR, "keypoints"), exist_ok=True)

NAME_RE = re.compile(r"subject_(\d+)_push_up_(good|bad)_(front|side|diagonal)\.mp4$")


def extract_video(video_path):
    options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
    )
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    kp2d, kp3d, detected = [], [], []
    # Mỗi video một landmarker mới: chế độ VIDEO giữ trạng thái tracking, timestamp phải tăng
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        i = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(i * 1000 / fps))
            i += 1
            if result.pose_landmarks:
                kp2d.append([[l.x, l.y, l.z, l.visibility] for l in result.pose_landmarks[0]])
                kp3d.append([[l.x, l.y, l.z, l.visibility] for l in result.pose_world_landmarks[0]])
                detected.append(True)
            else:
                kp2d.append(np.zeros((33, 4)))
                kp3d.append(np.zeros((33, 4)))
                detected.append(False)
    cap.release()
    return dict(
        kp2d=np.asarray(kp2d, dtype=np.float32).reshape(-1, 33, 4),
        kp3d=np.asarray(kp3d, dtype=np.float32).reshape(-1, 33, 4),
        detected=np.asarray(detected, dtype=bool),
        fps=np.float32(fps), width=np.int32(width), height=np.int32(height),
    )


rows = []
videos = sorted(glob.glob(os.path.join(PUSHUP_DIR, "**", "*.mp4"), recursive=True))
t0 = time.time()
for n, path in enumerate(videos, 1):
    m = NAME_RE.search(os.path.basename(path))
    if not m:
        print("Bỏ qua (tên lạ):", path)
        continue
    subject, quality, view = int(m.group(1)), m.group(2), m.group(3)
    out_name = os.path.basename(path).replace(".mp4", ".npz")
    out_path = os.path.join(OUT_DIR, "keypoints", out_name)

    if os.path.exists(out_path):
        d = np.load(out_path)
    else:
        d = extract_video(path)
        np.savez_compressed(out_path, **d)

    det = d["detected"]
    vis = d["kp2d"][det][:, :, 3] if det.any() else np.zeros((1, 33))
    rows.append(dict(
        file=out_name, subject=subject, quality=quality, view=view,
        fps=float(d["fps"]), width=int(d["width"]), height=int(d["height"]),
        n_frames=len(det), detect_rate=round(float(det.mean()), 3),
        vis_shoulder=round(float(vis[:, [11, 12]].max(1).mean()), 3),
        vis_elbow=round(float(vis[:, [13, 14]].max(1).mean()), 3),
        vis_wrist=round(float(vis[:, [15, 16]].max(1).mean()), 3),
        vis_hip=round(float(vis[:, [23, 24]].max(1).mean()), 3),
        vis_ankle=round(float(vis[:, [27, 28]].max(1).mean()), 3),
    ))
    print(f"[{n}/{len(videos)}] {out_name}  T={len(det)}  detect={det.mean():.2f}  ({time.time()-t0:.0f}s)")

index = pd.DataFrame(rows)
index.to_csv(os.path.join(OUT_DIR, "pushup_index.csv"), index=False)
json.dump({"mediapipe": mp.__version__, "model": "pose_landmarker_full (float16)"},
          open(os.path.join(OUT_DIR, "extract_info.json"), "w"))

## 6. Kiểm tra chất lượng keypoint

In [ ]:
cols = ["n_frames", "detect_rate", "vis_shoulder", "vis_elbow", "vis_wrist", "vis_hip", "vis_ankle"]
display(index.groupby(["view", "quality"])[cols].mean().round(2))

print("Video có tỉ lệ phát hiện < 0.9:")
display(index[index.detect_rate < 0.9][["file", "n_frames", "detect_rate"]])

## 7. Xem khung xương trên video

Vẽ khung xương ở điểm cao nhất và thấp nhất của một người, đủ 3 góc × good/bad.
Xương màu xanh: visibility > 0.6. Màu đỏ: visibility thấp, không đáng tin.

In [ ]:
import matplotlib.pyplot as plt

SUBJECT = 4   # đổi số này để xem người khác

EDGES = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (11, 23), (12, 24), (23, 24),
         (23, 25), (25, 27), (24, 26), (26, 28), (27, 31), (28, 32)]


def joint_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


fig, axes = plt.subplots(2, 6, figsize=(24, 10))
for r, quality in enumerate(["good", "bad"]):
    for c, view in enumerate(["front", "side", "diagonal"]):
        name = f"subject_{SUBJECT:03d}_push_up_{quality}_{view}"
        d = np.load(os.path.join(OUT_DIR, "keypoints", name + ".npz"))
        kp = d["kp2d"]
        px = kp[:, :, :2] * np.array([int(d["width"]), int(d["height"])])   # về pixel để góc không bị méo

        # Góc khuỷu của bên tay nhìn rõ hơn
        side = 0 if kp[:, [11, 13, 15], 3].mean() >= kp[:, [12, 14, 16], 3].mean() else 1
        elbow = joint_angle(px[:, 11 + side], px[:, 13 + side], px[:, 15 + side])
        elbow[~d["detected"]] = np.nan

        cap = cv2.VideoCapture(os.path.join(PUSHUP_DIR, quality, view, name + ".mp4"))
        for k, fi in enumerate([np.nanargmax(elbow), np.nanargmin(elbow)]):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ok, frame = cap.read()
            ax = axes[r, c * 2 + k]
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            p, vis = px[fi], kp[fi, :, 3]
            for a, b in EDGES:
                ax.plot([p[a, 0], p[b, 0]], [p[a, 1], p[b, 1]], lw=2,
                        color="lime" if min(vis[a], vis[b]) > 0.6 else "red")
            ax.scatter(p[11:, 0], p[11:, 1], s=12, c="yellow")
            ax.set_title(f"{quality}/{view}  khung {fi}  khuỷu={elbow[fi]:.0f}°")
            ax.axis("off")
        cap.release()
plt.tight_layout()
plt.show()

## 8. Nén kết quả để tải về

`/kaggle/working` bị xoá khi hết phiên. Tải `pushup_keypoints.zip` về (hoặc Save Version) rồi tạo Kaggle Dataset từ nó,
các bước sau chỉ cần đọc `.npz`, không cần video.

In [ ]:
!cd "{WORK_DIR}" && zip -qr pushup_keypoints.zip pushup_keypoints
!ls -lh "{WORK_DIR}/pushup_keypoints.zip"